# Full tournament with an LLM judge

This notebook drove the collection of all seven verdict matrices in
`data/matrices/` (a cleaned-up version of the authors' notebook; the collection
logic is in `collector.py`).

- **Phase A** puts every pair of objects to the judge: n(n−1)/2 pairs, one call
  per pair in a random presentation order or two calls (both orders). Every
  verdict is appended to a crash-safe store, so an interrupted run resumes
  where it stopped: just run the cell again.
- **Phase B** makes no model calls. It builds the reference rankings on the
  full matrix (Copeland, win rate, Bradley–Terry) and replays heapsort and
  quicksort (three pivot seeds) on it, reporting Spearman's ρ to the Copeland
  reference and the number of comparisons each sort used.

| Preset | Instruction | Presentation | Top-level note | Objects | Judge |
|---|---|---|---|---|---|
| GA2 | auth_v3 (authenticity, ru) | both orders | yes | Russian corpus* | Gemma4:26 |
| GA1 | auth_v3 (authenticity, ru) | one call, random order | yes | Russian corpus* | Gemma4:26 |
| GR1 | relevance_v2 (relevance, ru) | one call, random order | yes | Russian corpus* | Gemma4:26 |
| LR2 | relevance_v2 (relevance, ru) | both orders | yes | Russian corpus* | llama3.1:8b |
| GA2_en_ru | auth_v3 (authenticity, ru) | both orders | yes | `data/corpus_400-2_en.csv` | Gemma4:26 |
| GA2_en | auth_eng_v3 (authenticity, en) | both orders | yes | `data/corpus_400-2_en.csv` | Gemma4:26 |
| PDD | coh_eng_v1 (coherence, en) | both orders | no | `data/pdd_context_200.csv` | Gemma4:26 |

\* The Russian original of the corpus is not published, so these four presets
cannot be collected again; Phase B still runs on their published matrices
(`ANALYZE = "published"`). For PDD, build `data/pdd_context_200.csv` with
`prepare_pdd.py` first. The instruction texts are in `prompts/`.

**Serving the judge (Phase A only).** Gemma4:26 is the label of
gemma-4-26B-A4B-it-GGUF:IQ4_XS served by llama.cpp's `llama-server` with its
OpenAI-compatible endpoint (for example `llama-server -m <model.gguf> --port
8080`). llama3.1:8b-instruct-q4_K_M runs in Ollama (`ollama pull
llama3.1:8b-instruct-q4_K_M`). All runs used temperature 0, one pair per call
and at most 8 answer tokens. On our hardware (Section 2.8 of the article) a
tournament over 399 objects took about 8 hours with one call per pair and
about 16 hours with both orders. Start with a small `N_ROWS` to test the setup.


In [ ]:
import datetime
import os

import pandas as pd

import collector as pc
from ranklib import (bt_ranking, bt_strengths_se, copeland_ranking,
                     nontransitivity_exact, nontransitivity_sampled,
                     ranks_from_order, run_sort, spearman, winrate_ranking)

# Any order_mode other than "bidirectional" means ONE call per pair with a random
# presentation order; the order_mode names are those of the original runs.
RU = "data/corpus_400-2_ru.csv"      # not published
EN = "data/corpus_400-2_en.csv"
PDD = "data/pdd_context_200.csv"     # built by prepare_pdd.py
GEMMA = dict(backend="openai_local", model="Gemma4:26")
LLAMA = dict(backend="ollama", model="llama3.1:8b-instruct-q4_K_M")
PRESETS = {
    "GA2":       dict(prompt_version="auth_v3",      order_mode="bidirectional",        root_clause=True,  input_csv=RU,  **GEMMA),
    "GA1":       dict(prompt_version="auth_v3",      order_mode="bidirectional_random", root_clause=True,  input_csv=RU,  **GEMMA),
    "GR1":       dict(prompt_version="relevance_v2", order_mode="unidirectional",       root_clause=True,  input_csv=RU,  **GEMMA),
    "LR2":       dict(prompt_version="relevance_v2", order_mode="bidirectional",        root_clause=True,  input_csv=RU,  **LLAMA),
    "GA2_en_ru": dict(prompt_version="auth_v3",      order_mode="bidirectional",        root_clause=True,  input_csv=EN,  **GEMMA),
    "GA2_en":    dict(prompt_version="auth_eng_v3",  order_mode="bidirectional",        root_clause=True,  input_csv=EN,  **GEMMA),
    "PDD":       dict(prompt_version="coh_eng_v1",   order_mode="bidirectional",        root_clause=False, input_csv=PDD, **GEMMA),
}

PRESET = "GA2_en"          # one of PRESETS
COLLECT = True             # Phase A: False = skip the collection (no model needed)
ANALYZE = "collected"      # Phase B on "collected" (the store of Phase A) or "published"
N_ROWS = None              # None = all objects; e.g. 30 for a first test
FORCE_RERUN = False
OUT_DIR = "tournaments"    # stores, status files, run log, reports
SERVER_URL = "http://localhost:8080/v1/chat/completions"   # llama-server
# SERVER_URL = "http://192.168.88.11:8080/v1/chat/completions",  # llama-server -m model.gguf --port 8080
OLLAMA_URL = "http://localhost:11434/api/generate"

p = PRESETS[PRESET]
SETTINGS = pc.GlobalSettings(
    openai_local_url=SERVER_URL, ollama_url=OLLAMA_URL,
    runs_log_path=os.path.join(OUT_DIR, "runs_log.jsonl"),
    fail_log=os.path.join(OUT_DIR, "failures.log"),
    script_version="tournament-1.0")
cfg = pc.make_tournament_config(p["backend"], p["model"], p["prompt_version"],
                                p["order_mode"], input_csv=p["input_csv"],
                                out_dir=OUT_DIR, root_clause=p["root_clause"],
                                n_rows=N_ROWS, force_rerun=FORCE_RERUN)
print("exp_id:", cfg.exp_id)
print("store :", cfg.cache_path)
pc.list_instructions()


## Phase A — collect the complete matrix (model calls)


In [ ]:
if COLLECT:
    tour = pc.Tournament(cfg, SETTINGS)
    print(f"Loaded {tour.n_rows} objects; {tour.n_root_rows} top-level.")
    record = tour.collect_all_pairs(progress=True)
else:
    print("Phase A skipped")


## Phase B — reference rankings and sort replays (no model calls)


In [ ]:
path = (cfg.cache_path if ANALYZE == "collected"
        else os.path.join("data", "matrices", f"{PRESET}.jsonl"))
matrix = pc.load_matrix(path)
row_ids = pc.matrix_objects(matrix)
win = pc.win_function(matrix)
total_pairs = len(row_ids) * (len(row_ids) - 1) // 2

# reference rankings from the full matrix
cop_scores, cop_ranks, cop_ordered = copeland_ranking(matrix, row_ids)
win_scores, win_ranks, _ = winrate_ranking(matrix, row_ids)
bt_scores, bt_ranks, _ = bt_ranking(matrix, row_ids)
methods = {"copeland": cop_ranks, "winrate": win_ranks, "bt": bt_ranks}
costs = {m: len(matrix._data) for m in methods}

# replays of the sorts: every comparison is a lookup in the matrix
ordered, n_cmp = run_sort("heapsort", win, row_ids)
methods["heapsort"], costs["heapsort"] = ranks_from_order(ordered), n_cmp
for seed in (0, 1, 2):
    ordered, n_cmp = run_sort("quicksort", win, row_ids, seed=seed)
    methods[f"quicksort_s{seed}"] = ranks_from_order(ordered)
    costs[f"quicksort_s{seed}"] = n_cmp

nt_rate, nt_triples = nontransitivity_sampled(matrix, row_ids, sample=3000, seed=0)
nt_exact, nt_decisive, _ = nontransitivity_exact(matrix, row_ids)

names = list(methods)
lines = ["=" * 64,
         f"Reference bench — {os.path.basename(path)}",
         f"objects={len(row_ids)}  pairs={total_pairs}  matrix_records={len(matrix._data)}",
         f"instruction={p['prompt_version']}  order={p['order_mode']}  "
         f"top-level note={p['root_clause']}",
         f"non-transitivity rate = {nt_rate:.3f}  (sample of 3000 triples, "
         f"{nt_triples} decisive); exact = {nt_exact:.4f} over {nt_decisive} "
         f"decisive triples",
         "",
         "Spearman vs Copeland reference  |  comparisons used  |  coverage of full matrix"]
for nm in names:
    rho = spearman(cop_ranks, methods[nm])
    cov = costs[nm] / max(1, total_pairs)
    lines.append(f"  {nm:14} rho={rho:+.4f}   comparisons={costs[nm]:>7}   coverage={cov:6.1%}")
lines += ["", "Method x method Spearman matrix:",
          f"{'':14}" + "".join(f"{nm[:10]:>12}" for nm in names)]
for a in names:
    lines.append(f"{a:14}" + "".join(f"{spearman(methods[a], methods[b]):>12.3f}"
                                     for b in names))
lines.append("=" * 64)
report = "\n".join(lines)
print(report)

os.makedirs(OUT_DIR, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
rep_path = os.path.join(OUT_DIR, f"reference_bench_{PRESET}_{stamp}.txt")
with open(rep_path, "w", encoding="utf-8") as f:
    f.write(report)
print("\nReport saved to:", rep_path)


## Save the reference ranking

With the input file available, the ranking is saved together with the texts;
otherwise only the object ids and the scores are saved.


In [ ]:
ranking = pd.DataFrame({"object": row_ids})
ranking["copeland_score"] = ranking.object.map(cop_scores)
ranking["copeland_rank"] = ranking.object.map(cop_ranks)
ranking["winrate_score"] = ranking.object.map(win_scores)
ranking["winrate_rank"] = ranking.object.map(win_ranks)
ranking["bt_score"] = ranking.object.map(bt_scores)       # theta = log-strength
ranking["bt_rank"] = ranking.object.map(bt_ranks)
ranking["bt_se"] = ranking.object.map(bt_strengths_se(matrix, row_ids))

if os.path.exists(p["input_csv"]) and N_ROWS is None:
    texts = pc.load_dataset(cfg)
    if len(texts) == len(row_ids):
        ranking = pd.concat([texts, ranking.drop(columns="object")], axis=1)
        ranking = pc.flatten_text_cells(ranking) if SETTINGS.flatten_output else ranking
out_csv = os.path.join(OUT_DIR, f"reference_ranking_{PRESET}_{ANALYZE}.csv")
ranking.to_csv(out_csv, index=False, encoding="utf-8-sig", float_format="%.6f", na_rep="")
print("Reference ranking saved to:", out_csv)


## Optional: compare a new collection with the published matrix

Share of pairs with the same aggregated verdict, the tie rates, and Spearman's
ρ between the two Copeland reference rankings.


In [ ]:
published = os.path.join("data", "matrices", f"{PRESET}.jsonl")
if os.path.exists(cfg.cache_path) and os.path.exists(published):
    a, b = pc.load_matrix(cfg.cache_path), pc.load_matrix(published)
    common = sorted(set(a._data) & set(b._data))
    same = sum(a._data[k]["winner"] == b._data[k]["winner"] for k in common)
    ties_a = sum(a._data[k]["winner"] is None for k in common)
    ties_b = sum(b._data[k]["winner"] is None for k in common)
    ids = pc.matrix_objects(b)
    if len(common) == len(ids) * (len(ids) - 1) // 2:
        rho = spearman(copeland_ranking(a, ids)[1], copeland_ranking(b, ids)[1])
        rho_txt = f"{rho:.4f}"
    else:
        rho_txt = "n/a (incomplete collection)"
    print(f"common pairs {len(common)}; same verdict {same / len(common):.1%}; "
          f"tie rate new {ties_a / len(common):.1%} vs published {ties_b / len(common):.1%}; "
          f"Copeland rho {rho_txt}")
else:
    print("nothing to compare: the collected store or the published matrix is missing")
